# 📘 Ficha Prática Tutorial: Máquina Virtual Didática EWVM (v2.0)
### **Universidade do Minho — Licenciatura em Engenharia Informática / Ciências da Computação**
**Unidade Curricular:** Processamento de Linguagens / Compiladores (PLC)  
**Ambiente Oficial:** [https://ewvm.epl.di.uminho.pt/](https://ewvm.epl.di.uminho.pt/)  
**Autor:** Prof. José Carlos Ramalho 

---

## 🎯 Objetivos Pedagógicos
Esta ficha tutorial tem como objetivo guiar os estudantes na **compreensão profunda e prática da geração e execução de código para uma Máquina de Pilha (Stack Machine)**.

Ao longo deste tutorial, irá:
1. Compreender a arquitetura interna da **EWVM** (registos `PC`, `SP`, `FP`, `GP`, stacks e heaps).
2. Dominar o conjunto completo de instruções Assembly da EWVM.
3. Desenvolver algoritmos organizados por ordem crescente de complexidade:
   - **Nível 1:** Aritmética elementar, expressões pós-fixas e I/O sequencial.
   - **Nível 2:** Estruturas de controlo condicional (`IF-THEN-ELSE`).
   - **Nível 3:** Estruturas de repetição e ciclos iterativos (`WHILE`, `FOR`).
   - **Nível 4:** Alocação dinâmica de memória, arrays e vetores na **Struct Heap**.
   - **Nível 5:** Sub-rotinas, procedimentos, passagem de parâmetros e recursividade.
4. Executar e testar os seus programas tanto no **simulador Python embutido** neste notebook como diretamente na **plataforma Web oficial via API**.


---
## 🏛️ 1. Arquitetura da Máquina Virtual EWVM

A **EWVM** é uma máquina virtual abstrata baseada numa arquitetura de stack. Ao contrário das arquiteturas baseadas em registos de uso geral (como x86-64 ou ARM), todas as operações aritméticas e lógicas retiram os seus operandos do topo de uma stack de operandos e empilham o resultado de volta.

```
                    ┌────────────────────────────────────────┐
                    │               E W V M                  │
                    │                                        │
  Registos:         │   [ PC ]     [ SP ]   [ FP ]   [ GP ]  │
                    └─────┬──────────┬────────┬────────┬─────┘
                          │          │        │        │
      ┌───────────────────┴──┐       │        │        │
      │  Memória de Código   │       │        │        │
      │  (Instruções & Tags) │       │        │        │
      └──────────────────────┘       ▼        ▼        ▼
                              ┌─────────────────────────────────────────────┐
                              │     Operand Stack (Pilha)                   │
                              │   - Variáveis Globais (gp - global pointer) │
                              │   - Frames Locais (fp - frame pointer)      │
                              │   - Avaliação Temporária                    │
                              └──────────────┬──────────────────────────────┘
                                             │
      ┌──────────────────────┐               │               ┌──────────────────────┐
      │      Call Stack      │◄──────────────┴──────────────►│     String Heap      │
      │ (End. Retorno + FP)  │                               │ (Cadeias de Caract.) │
      └──────────────────────┘                               └──────────────────────┘
                                             ▲
                                             │
                              ┌──────────────┴───────────────┐
                              │         Struct Heap          │
                              │  (Blocos Dinâmicos / Arrays) │
                              └──────────────────────────────┘
```

### 1.1 Registos Principais
* **`PC` (Program Counter):** Aponta para o endereço da instrução atualmente em execução.
* **`SP` (Stack Pointer):** Aponta para o topo atual da pilha de operandos.
* **`FP` (Frame Pointer):** Aponta para a base da pilha de ativação local da função corrente. Antes do `START` ou fora de funções, está indefinido ou aponta para a base global.
* **`GP` (Global Pointer):** Aponta para a base das variáveis globais (`gp = 0`).

### 1.2 Espaços de Memória
1. **Operand Stack (Pilha de Operandos):** Pilha LIFO onde os cálculos ocorrem e onde as variáveis globais e locais residem.
   - **Variáveis Globais:** São alocadas no início do programa antes da instrução `START` (por exemplo com `pushi 0` ou `pushn 5`). São indexadas através de `pushg n` e `storeg n` relativamente a `GP`.
   - **Variáveis Locais:** Alocadas após a entrada numa função e indexadas através de `pushl n` e `storel n` relativamente a `FP`.
2. **Call Stack (Pilha de Chamadas):** Utilizada pelas instruções `CALL` e `RETURN` para guardar o par `(PC, FP)` de retorno ao chamar sub-rotinas.
3. **String Heap:** Guarda todas as cadeias de caracteres literais (`PUSHS "texto"`) e lidas pelo utilizador (`READ`). A pilha de operandos apenas manipula apontadores (endereços de string) para esta heap.
4. **Struct Heap (Heap de Estruturas / Arrays):** Bloco de memória para alocação dinâmica (`ALLOC` e `ALLOCN`). Permite armazenar vetores e estruturas indexadas por endereços e desvios (`LOAD`, `LOADN`, `STORE`, `STOREN`).

### 1.3 Esqueleto Canónico de um Programa EWVM
Um programa EWVM é dividido tipicamente em três fases:
1. **Reserva de Variáveis Globais:** Declarações com `pushi` / `pushn` / `pushf`.
2. **Ponto de Entrada (`START`):** Instrução que inicializa a máquina e define `FP = SP`.
3. **Corpo de Execução e Sub-rotinas:** Instruções de processamento, etiquetas (`labels:`) e terminação com `STOP`.

```assembly
// 1. Alocação de variáveis globais: gp[0], gp[1]
pushi 0    // gp[0] (ex: variável 'x')
pushi 0    // gp[1] (ex: variável 'y')

// 2. Início do programa principal
start

// 3. Código executável
pushs "Ola, EWVM!"
writes
writeln

// 4. Fim do programa
stop
```


---
## 📖 2. Tabela de Referência Rápida de Instruções (ISA EWVM 2.0)

| Categoria | Instrução | Argumentos | Descrição / Efeito na Pilha |
| :--- | :--- | :--- | :--- |
| **Aritmética Inteira** | `ADD` | - | Retira $n$ e $m$; empilha $m + n$ |
| | `SUB` | - | Retira $n$ e $m$; empilha $m - n$ |
| | `MUL` | - | Retira $n$ e $m$; empilha $m 	imes n$ |
| | `DIV` | - | Retira $n$ e $m$; empilha $m / n$ (divisão inteira) |
| | `MOD` | - | Retira $n$ e $m$; empilha $m \pmod n$ |
| | `NOT` | - | Retira $n$; empilha $1$ se $n = 0$, senão $0$ |
| **Comparações** | `INF` | - | Retira $n$ e $m$; empilha $1$ se $m < n$, senão $0$ |
| | `INFEQ` | - | Retira $n$ e $m$; empilha $1$ se $m \le n$, senão $0$ |
| | `SUP` | - | Retira $n$ e $m$; empilha $1$ se $m > n$, senão $0$ |
| | `SUPEQ` | - | Retira $n$ e $m$; empilha $1$ se $m \ge n$, senão $0$ |
| | `EQUAL` | - | Retira $n$ e $m$; empilha $1$ se $m = n$, senão $0$ |
| **Lógica Booleana** | `AND` | - | Retira $n$ e $m$; empilha $1$ se $m 
e 0 \land n 
e 0$, senão $0$ |
| | `OR` | - | Retira $n$ e $m$; empilha $1$ se $m 
e 0 \lor n 
e 0$, senão $0$ |
| **Vírgula Flutuante** | `FADD`, `FSUB` | - | Adição / Subtração de números reais |
| | `FMUL`, `FDIV` | - | Multiplicação / Divisão de números reais |
| | `FCOS`, `FSIN` | - | Cosseno e Seno |
| | `FINF`, `FINFEQ` | - | Comparações reais ($<$, $\le$) |
| | `FSUP`, `FSUPEQ` | - | Comparações reais ($>$, $\ge$) |
| **Conversões** | `ATOI` / `ATOF` | - | Converte string na String Heap para Inteiro / Real |
| | `ITOF` / `FTOI` | - | Converte Inteiro para Real / Real para Inteiro |
| | `STRI` / `STRF` | - | Converte Inteiro / Real para string e empilha endereço |
| **Empilhamento** | `PUSHI n` | Inteiro | Empilha a constante inteira $n$ |
| | `PUSHN n` | Inteiro | Empilha $n$ vezes o inteiro $0$ |
| | `PUSHF f` | Real | Empilha a constante real $f$ |
| | `PUSHS "str"` | String | Armazena na String Heap e empilha endereço |
| | `PUSHG n` | Inteiro | Empilha o valor em `gp[n]` |
| | `PUSHL n` | Inteiro | Empilha o valor em `fp[n]` |
| | `PUSHSP` / `PUSHFP` / `PUSHGP` | - | Empilha o valor dos registradores `SP`, `FP` ou `GP` |
| **Manipulação de Pilha** | `DUP [n]` | [Inteiro] | Duplica o elemento do topo da pilha $n$ vezes |
| | `DUPN` | - | Retira $n$ e duplica o novo topo $n$ vezes |
| | `COPY n` | Inteiro | Copia os $n$ elementos do topo e empilha na mesma ordem |
| | `COPYN` | - | Retira $n$ e copia os $n$ elementos do topo |
| | `POP [n]` | [Inteiro] | Remove $n$ elementos do topo |
| | `POPN` | - | Retira $n$ e remove $n$ elementos do topo |
| | `SWAP` | - | Troca os dois elementos do topo |
| | `CHECK n, p` | $n, p$ | Valida se $n \le 	ext{topo} \le p$ (lança erro se falso) |
| **Armazenamento** | `STOREG n` | Inteiro | Retira valor do topo e guarda em `gp[n]` |
| | `STOREL n` | Inteiro | Retira valor do topo e guarda em `fp[n]` |
| | `STORE n` | Inteiro | Retira endereço $a$ e valor $v$; guarda $v$ em $a[n]$ |
| | `STOREN` | - | Retira endereço $a$, índice $n$ e valor $v$; guarda $v$ em $a[n]$ |
| | `LOAD n` | Inteiro | Retira endereço $a$; empilha valor em $a[n]$ |
| | `LOADN` | - | Retira endereço $a$ e índice $n$; empilha valor em $a[n]$ |
| **Heap Dinâmica** | `ALLOC n` | Inteiro | Aloca bloco de tamanho $n$ e empilha endereço base |
| | `ALLOCN` | - | Retira tamanho $n$; aloca bloco de tamanho $n$ |
| | `FREE` | - | Retira endereço e liberta o bloco alocado |
| **Input / Output** | `READ` | - | Lê string do utilizador, guarda na String Heap e empilha endereço |
| | `WRITEI` | - | Retira inteiro do topo e imprime |
| | `WRITEF` | - | Retira real do topo e imprime |
| | `WRITES` | - | Retira endereço de string e imprime |
| | `WRITELN` | - | Imprime mudança de linha (`
`) |
| | `WRITECHR` | - | Retira inteiro do topo e imprime o caractere ASCII correspondente |
| **Controlo de Fluxo** | `PUSHA label`| Etiqueta | Empilha o endereço de código de `label` |
| | `JUMP label` | Etiqueta | Salto incondicional para `label` |
| | `JZ label` | Etiqueta | Retira $v$; salta se $v = 0$, senão continua |
| | `CALL` | - | Retira endereço da função, guarda contexto na Call Stack e salta |
| | `RETURN` | - | Restaura contexto da Call Stack e retorna ao chamador |
| | `START` | - | Define início do programa (`fp = sp`) |
| | `STOP` | - | Termina a execução |


---
## 💻 3. Ambiente de Execução Python Integrado

Para permitir a resolução interativa dos exercícios diretamente neste notebook, disponibilizamos duas formas de execução:
1. **`EWVMInterpreter`**: Emulador local completo em Python (permite execução offline e inspeção detalhada da pilha passo a passo).
2. **`ewvm_remote_run`**: Chamada à API Web oficial da EWVM (`https://ewvm.epl.di.uminho.pt/api/run`) para submissão direta ao servidor.

Execute a célula seguinte para inicializar o ambiente:


In [ ]:
import re
import json
import urllib.request
import urllib.error

class EWVMInterpreter:
    """
    Emulador Didático da Máquina Virtual EWVM em Python Puro.
    Suporta todas as instruções da especificação EWVM 2.0.
    """
    def __init__(self, code, inputs=None, trace=False):
        self.raw_code = code
        self.inputs = list(inputs) if inputs else []
        self.input_idx = 0
        self.trace = trace
        self.output = []
        self.op_stack = []
        self.call_stack = []
        self.string_heap = []
        self.struct_heap = {}
        self.heap_counter = 1000
        self.gp = 0
        self.fp = -1
        self.pc = 0
        self.instructions = []
        self.labels = {}
        self._parse()

    def _parse(self):
        lines = self.raw_code.splitlines()
        for line_no, line in enumerate(lines, start=1):
            clean = re.sub(r'//.*', '', line).strip()
            if not clean:
                continue
            # Processar labels
            while ':' in clean:
                parts = clean.split(':', 1)
                lbl = parts[0].strip()
                if ' ' not in lbl and '\t' not in lbl:
                    self.labels[lbl] = len(self.instructions)
                    clean = parts[1].strip()
                else:
                    break
            if not clean:
                continue
            tokens = clean.split(None, 1)
            op = tokens[0].upper()
            arg = tokens[1].strip() if len(tokens) > 1 else None
            self.instructions.append((op, arg, line_no, clean))

    def _resolve_mem(self, addr, offset):
        if addr in self.struct_heap:
            return self.struct_heap[addr], offset
        else:
            return self.op_stack, addr + offset

    def _get_str(self, ptr):
        if isinstance(ptr, str) and ptr.startswith("string#"):
            idx = int(ptr.split("#")[1])
            return self.string_heap[idx]
        elif isinstance(ptr, int) and 0 <= ptr < len(self.string_heap):
            return self.string_heap[ptr]
        return str(ptr)

    def run(self, max_steps=500000):
        steps = 0
        while self.pc < len(self.instructions) and steps < max_steps:
            steps += 1
            op, arg, line_no, raw = self.instructions[self.pc]

            if self.trace:
                print(f"[PC={self.pc:03d} L{line_no:02d}] {raw:<25} | Stack: {self.op_stack} | FP={self.fp}")

            if op == 'START':
                self.fp = len(self.op_stack)
                self.pc += 1
            elif op == 'STOP':
                break
            elif op == 'NOP':
                self.pc += 1
            elif op == 'ERR':
                msg = arg.strip('"') if arg else "Erro de execução"
                self.output.append(f"\n[ERRO EWVM]: {msg}\n")
                break
            elif op == 'PUSHI':
                self.op_stack.append(int(arg))
                self.pc += 1
            elif op == 'PUSHN':
                n = int(arg)
                self.op_stack.extend([0] * n)
                self.pc += 1
            elif op == 'PUSHF':
                self.op_stack.append(float(arg))
                self.pc += 1
            elif op == 'PUSHS':
                s = arg.strip()
                if s.startswith('"') and s.endswith('"'):
                    s = s[1:-1].encode().decode('unicode_escape')
                idx = len(self.string_heap)
                self.string_heap.append(s)
                self.op_stack.append(f"string#{idx}")
                self.pc += 1
            elif op == 'PUSHG':
                idx = int(arg)
                self.op_stack.append(self.op_stack[self.gp + idx])
                self.pc += 1
            elif op == 'STOREG':
                idx = int(arg)
                val = self.op_stack.pop()
                self.op_stack[self.gp + idx] = val
                self.pc += 1
            elif op == 'PUSHL':
                idx = int(arg)
                self.op_stack.append(self.op_stack[self.fp + idx])
                self.pc += 1
            elif op == 'STOREL':
                idx = int(arg)
                val = self.op_stack.pop()
                self.op_stack[self.fp + idx] = val
                self.pc += 1
            elif op == 'PUSHSP':
                self.op_stack.append(len(self.op_stack))
                self.pc += 1
            elif op == 'PUSHFP':
                self.op_stack.append(self.fp)
                self.pc += 1
            elif op == 'PUSHGP':
                self.op_stack.append(self.gp)
                self.pc += 1
            elif op == 'DUP':
                n = int(arg) if arg else 1
                val = self.op_stack[-1]
                self.op_stack.extend([val] * n)
                self.pc += 1
            elif op == 'DUPN':
                n = self.op_stack.pop()
                val = self.op_stack[-1]
                self.op_stack.extend([val] * n)
                self.pc += 1
            elif op == 'COPY':
                n = int(arg)
                vals = self.op_stack[-n:]
                self.op_stack.extend(vals)
                self.pc += 1
            elif op == 'COPYN':
                n = self.op_stack.pop()
                vals = self.op_stack[-n:]
                self.op_stack.extend(vals)
                self.pc += 1
            elif op == 'POP':
                n = int(arg) if arg else 1
                for _ in range(n):
                    self.op_stack.pop()
                self.pc += 1
            elif op == 'POPN':
                n = self.op_stack.pop()
                for _ in range(n):
                    self.op_stack.pop()
                self.pc += 1
            elif op == 'SWAP':
                v = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(v)
                self.op_stack.append(m)
                self.pc += 1
            elif op == 'ADD':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(m + n)
                self.pc += 1
            elif op == 'SUB':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(m - n)
                self.pc += 1
            elif op == 'MUL':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(m * n)
                self.pc += 1
            elif op == 'DIV':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(m // n)
                self.pc += 1
            elif op == 'MOD':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(m % n)
                self.pc += 1
            elif op == 'NOT':
                n = self.op_stack.pop()
                self.op_stack.append(1 if n == 0 else 0)
                self.pc += 1
            elif op == 'INF':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if m < n else 0)
                self.pc += 1
            elif op == 'INFEQ':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if m <= n else 0)
                self.pc += 1
            elif op == 'SUP':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if m > n else 0)
                self.pc += 1
            elif op == 'SUPEQ':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if m >= n else 0)
                self.pc += 1
            elif op == 'EQUAL':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if m == n else 0)
                self.pc += 1
            elif op == 'AND':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if (m != 0 and n != 0) else 0)
                self.pc += 1
            elif op == 'OR':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if (m != 0 or n != 0) else 0)
                self.pc += 1
            elif op == 'FADD':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(float(m) + float(n))
                self.pc += 1
            elif op == 'FSUB':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(float(m) - float(n))
                self.pc += 1
            elif op == 'FMUL':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(float(m) * float(n))
                self.pc += 1
            elif op == 'FDIV':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(float(m) / float(n))
                self.pc += 1
            elif op == 'FINF':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if float(m) < float(n) else 0)
                self.pc += 1
            elif op == 'FINFEQ':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if float(m) <= float(n) else 0)
                self.pc += 1
            elif op == 'FSUP':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if float(m) > float(n) else 0)
                self.pc += 1
            elif op == 'FSUPEQ':
                n = self.op_stack.pop()
                m = self.op_stack.pop()
                self.op_stack.append(1 if float(m) >= float(n) else 0)
                self.pc += 1
            elif op == 'ITOF':
                v = self.op_stack.pop()
                self.op_stack.append(float(v))
                self.pc += 1
            elif op == 'FTOI':
                v = self.op_stack.pop()
                self.op_stack.append(int(v))
                self.pc += 1
            elif op == 'ATOI':
                ptr = self.op_stack.pop()
                s = self._get_str(ptr)
                self.op_stack.append(int(s))
                self.pc += 1
            elif op == 'ATOF':
                ptr = self.op_stack.pop()
                s = self._get_str(ptr)
                self.op_stack.append(float(s))
                self.pc += 1
            elif op == 'STRI':
                v = self.op_stack.pop()
                idx = len(self.string_heap)
                self.string_heap.append(str(v))
                self.op_stack.append(f"string#{idx}")
                self.pc += 1
            elif op == 'STRF':
                v = self.op_stack.pop()
                idx = len(self.string_heap)
                self.string_heap.append(str(v))
                self.op_stack.append(f"string#{idx}")
                self.pc += 1
            elif op == 'CONCAT':
                m_ptr = self.op_stack.pop()
                n_ptr = self.op_stack.pop()
                ns = self._get_str(n_ptr)
                ms = self._get_str(m_ptr)
                idx = len(self.string_heap)
                self.string_heap.append(ns + ms)
                self.op_stack.append(f"string#{idx}")
                self.pc += 1
            elif op == 'STRLEN':
                ptr = self.op_stack.pop()
                s = self._get_str(ptr)
                self.op_stack.append(len(s))
                self.pc += 1
            elif op == 'CHRCODE':
                ptr = self.op_stack.pop()
                s = self._get_str(ptr)
                self.op_stack.append(ord(s[0]) if s else 0)
                self.pc += 1
            elif op == 'CHARAT':
                m_ptr = self.op_stack.pop()
                n = self.op_stack.pop()
                s = self._get_str(m_ptr)
                self.op_stack.append(ord(s[n]))
                self.pc += 1
            elif op == 'WRITEI':
                v = self.op_stack.pop()
                self.output.append(str(v))
                self.pc += 1
            elif op == 'WRITEF':
                v = self.op_stack.pop()
                self.output.append(str(v))
                self.pc += 1
            elif op == 'WRITES':
                ptr = self.op_stack.pop()
                self.output.append(self._get_str(ptr))
                self.pc += 1
            elif op == 'WRITELN':
                self.output.append("\n")
                self.pc += 1
            elif op == 'WRITECHR':
                v = self.op_stack.pop()
                self.output.append(chr(v))
                self.pc += 1
            elif op == 'READ':
                val = self.inputs[self.input_idx] if self.input_idx < len(self.inputs) else ""
                self.input_idx += 1
                idx = len(self.string_heap)
                self.string_heap.append(str(val))
                self.op_stack.append(f"string#{idx}")
                self.pc += 1
            elif op == 'PUSHA':
                self.op_stack.append(arg)
                self.pc += 1
            elif op == 'JUMP':
                self.pc = self.labels[arg]
            elif op == 'JZ':
                v = self.op_stack.pop()
                if v == 0:
                    self.pc = self.labels[arg]
                else:
                    self.pc += 1
            elif op == 'CALL':
                target = self.op_stack.pop()
                target_pc = self.labels[target] if isinstance(target, str) else target
                self.call_stack.append((self.pc, self.fp))
                self.fp = len(self.op_stack)
                self.pc = target_pc
            elif op == 'RETURN':
                ret_pc, old_fp = self.call_stack.pop()
                del self.op_stack[self.fp:]
                self.fp = old_fp
                self.pc = ret_pc + 1
            elif op == 'ALLOC':
                n = int(arg)
                addr = self.heap_counter
                self.heap_counter += 1
                self.struct_heap[addr] = [0] * n
                self.op_stack.append(addr)
                self.pc += 1
            elif op == 'ALLOCN':
                n = self.op_stack.pop()
                addr = self.heap_counter
                self.heap_counter += 1
                self.struct_heap[addr] = [0] * n
                self.op_stack.append(addr)
                self.pc += 1
            elif op == 'FREE':
                addr = self.op_stack.pop()
                self.struct_heap.pop(addr, None)
                self.pc += 1
            elif op == 'STORE':
                idx = int(arg)
                addr = self.op_stack.pop()
                val = self.op_stack.pop()
                mem, loc = self._resolve_mem(addr, idx)
                mem[loc] = val
                self.pc += 1
            elif op == 'STOREN':
                val = self.op_stack.pop()
                n = self.op_stack.pop()
                addr = self.op_stack.pop()
                mem, loc = self._resolve_mem(addr, n)
                mem[loc] = val
                self.pc += 1
            elif op == 'LOAD':
                idx = int(arg)
                addr = self.op_stack.pop()
                mem, loc = self._resolve_mem(addr, idx)
                self.op_stack.append(mem[loc])
                self.pc += 1
            elif op == 'LOADN':
                n = self.op_stack.pop()
                addr = self.op_stack.pop()
                mem, loc = self._resolve_mem(addr, n)
                self.op_stack.append(mem[loc])
                self.pc += 1
            elif op == 'CHECK':
                p1, p2 = [int(x.strip()) for x in arg.split(',')]
                top = self.op_stack[-1]
                if not (p1 <= top <= p2):
                    raise ValueError(f"CHECK violado: {p1} <= {top} <= {p2} e falso!")
                self.pc += 1
            else:
                raise ValueError(f"Instrucao desconhecida: {op} (linha {line_no})")

        return "".join(self.output)

def ewvm_run(code, inputs=None, trace=False):
    """Funcao de conveniencia para executar codigo no emulador local."""
    interpreter = EWVMInterpreter(code, inputs=inputs, trace=trace)
    out = interpreter.run()
    print(out, end="")
    return out

def ewvm_remote_run(code, inputs=None):
    """
    Executa o programa remotamente no servidor oficial da EWVM (https://ewvm.epl.di.uminho.pt/api/run).
    """
    try:
        init_req = urllib.request.Request('https://ewvm.epl.di.uminho.pt/api/init')
        with urllib.request.urlopen(init_req, timeout=5) as r:
            init_data = json.loads(r.read().decode('utf-8'))
            session_id = init_data['data']['sessionId']

        payload = {'code': code, 'sessionId': session_id}
        inputs = list(inputs) if inputs else []
        inp_idx = 0

        while True:
            post_req = urllib.request.Request(
                'https://ewvm.epl.di.uminho.pt/api/run',
                data=json.dumps(payload).encode('utf-8'),
                headers={'Content-Type': 'application/json'}
            )
            with urllib.request.urlopen(post_req, timeout=5) as r:
                res = json.loads(r.read().decode('utf-8'))

            if not res.get('success'):
                print("Erro na resposta do servidor EWVM:", res)
                return

            term = res['data'].get('terminal', [])
            if res['data'].get('input') == 1 and inp_idx < len(inputs):
                cur_in = str(inputs[inp_idx])
                inp_idx += 1
                anim = res['data'].get('animation', [])
                payload = {
                    'code': code,
                    'sessionId': session_id,
                    'input': cur_in,
                    'index': len(anim) - 1 if anim else 0,
                    'terminal': term + [f"<< {cur_in}"]
                }
            else:
                out_str = "".join(term)
                print(out_str, end="")
                return out_str
    except Exception as e:
        print(f"Nota: Conexao a API remota indisponivel ({e}). A usar o emulador local:")
        return ewvm_run(code, inputs)

print("✅ Ambiente EWVM inicializado com sucesso!")


---
## 🟢 Nível 1: Aritmética Elementar, Notação Pós-Fixada e I/O Sequencial

### 🧠 Fundamento Teórico: Expressões em Máquinas de Pilha
Na EWVM, a avaliação de uma expressão aritmética corresponde diretamente à sua tradução em **Notação Polaca Invertida (RPN / Pós-fixada)**:
* Para avaliar $A + B$: empilha-se $A$, empilha-se $B$, executa-se `ADD`.
* Para operadores não comutativos como a subtração ($m - n$) e a divisão ($m / n$), a instrução retira primeiro o topo $n$ e depois o sub-topo $m$, empilhando $m - n$.
* A leitura de dados é feita com `READ` (devolve endereço na String Heap) seguida de `ATOI` (para inteiro) ou `ATOF` (para float).
* A escrita utiliza `WRITEI` (inteiro), `WRITEF` (real), `WRITES` (string) e `WRITELN` (`\n`).

---
### 📌 Exemplo Demonstrativo 1: Expressão Pós-Fixada
**Problema:** Calcular e imprimir o resultado da expressão:
$$\text{resultado} = ((15 + 25) \times 3) - (50 / 2)$$


In [ ]:
exemplo_1 = """
start
    // 1. Calcular (15 + 25) * 3
    pushi 15
    pushi 25
    add
    pushi 3
    mul

    // 2. Calcular 50 / 2
    pushi 50
    pushi 2
    div

    // 3. Subtrair os dois resultados parciais
    sub

    // 4. Imprimir o resultado final
    pushs "Resultado de ((15+25)*3) - (50/2) = "
    writes
    writei
    writeln
stop
"""

ewvm_run(exemplo_1, trace=True)


---
### ✏️ Exercício 1.1: Perímetro e Área de um Retângulo
**Objetivo:** Ler do utilizador o comprimento da **base** ($b$) e da **altura** ($h$) de um retângulo, calcular e imprimir:
* $\text{Perímetro} = 2 \times (b + h)$
* $\text{Área} = b \times h$

**Entrada esperada:** Dois inteiros fornecidos via teclado (`READ` + `ATOI`).  
**Variáveis Globais sugeridas:**
* `gp[0]`: base ($b$)
* `gp[1]`: altura ($h$)


In [ ]:
# Espaço para a resolução do Exercício 1.1:
exercicio_1_1 = """
// Alocação de variáveis globais
pushi 0    // gp[0]: base
pushi 0    // gp[1]: altura

start
    // 1. Ler base
    pushs "Introduza a base: "
    writes
    read
    atoi
    storeg 0

    // 2. Ler altura
    pushs "Introduza a altura: "
    writes
    read
    atoi
    storeg 1

    // 3. Calcular e imprimir o perímetro: 2 * (b + h)
    pushs "Perimetro: "
    writes
    pushi 2
    pushg 0
    pushg 1
    add
    mul
    writei
    writeln

    // 4. Calcular e imprimir a área: b * h
    pushs "Area: "
    writes
    pushg 0
    pushg 1
    mul
    writei
    writeln
stop
"""

# Testar com base=7 e altura=4 (Perímetro esperado = 22, Área esperada = 28)
ewvm_run(exercicio_1_1, inputs=["7", "4"], trace=True)


---
### ✏️ Exercício 1.2: Conversor de Temperatura (Celsius para Fahrenheit)
**Objetivo:** Ler uma temperatura inteira em graus Celsius ($C$), convertê-la para vírgula flutuante (`ITOF`) e calcular a temperatura correspondente em Fahrenheit ($F$) utilizando as instruções de vírgula flutuante:
$$F = C \times 1.8 + 32.0$$

**Instruções a praticar:** `ITOF`, `PUSHF`, `FMUL`, `FADD`, `WRITEF`.


In [ ]:
# Espaço para a resolução do Exercício 1.2:
exercicio_1_2 = """
pushi 0    // gp[0]: Celsius

start
    pushs "Temperatura em Celsius (inteiro): "
    writes
    read
    atoi
    storeg 0

    pushs "Temperatura em Fahrenheit: "
    writes
    
    // Obter Celsius e converter para Float
    pushg 0
    itof
    
    // Multiplicar por 1.8
    pushf 1.8
    fmul
    
    // Somar 32.0
    pushf 32.0
    fadd
    
    // Imprimir resultado real
    writef
    writeln
stop
"""

# Testar com C = 25 (Fahrenheit esperado = 77.0)
ewvm_run(exercicio_1_2, inputs=["25"])


---
### ✏️ Exercício 1.3: Manipulação de Pilha com `SWAP`, `DUP` e `POP`
**Objetivo:** Sem utilizar qualquer variável global (`storeg`/`pushg`), ler um número inteiro $x$ da entrada e calcular $2 \times x^2 + 5 \times x$ manipulando apenas o topo da pilha com `DUP` e operadores aritméticos!


In [ ]:
# Espaço para a resolução do Exercício 1.3:
exercicio_1_3 = """
start
    pushs "Introduza x: "
    writes
    read
    atoi       // Stack: [x]

    // Queremos calcular 2*x^2 + 5*x
    // Duplicar x para podermos calcular parcelas
    dup 1      // Stack: [x, x]
    dup 1      // Stack: [x, x, x]
    mul        // Stack: [x, x^2]
    pushi 2
    mul        // Stack: [x, 2*x^2]
    swap       // Stack: [2*x^2, x]
    pushi 5
    mul        // Stack: [2*x^2, 5*x]
    add        // Stack: [2*x^2 + 5*x]

    pushs "Resultado de 2*x^2 + 5*x = "
    writes
    writei
    writeln
stop
"""

# Testar com x = 3: 2*(9) + 5*(3) = 18 + 15 = 33
ewvm_run(exercicio_1_3, inputs=["3"])


---
## 🟡 Nível 2: Estruturas de Seleção Condicional (`IF-THEN-ELSE`)

### 🧠 Fundamento Teórico: Saltos Condicionais em Máquinas de Pilha
Em linguagens de alto nível, escrevemos:
```c
if (condicao) {
    bloco_then;
} else {
    bloco_else;
}
```

Na **EWVM**, este padrão traduz-se através da instrução `JZ label` (**Jump if Zero**):
1. Avalia-se a expressão da condição, deixando $1$ (verdadeiro) ou $0$ (falso) no topo da pilha.
2. Executa-se `JZ label_else`:
   - Se a condição for $0$ (falsa), salta para `label_else`.
   - Se for $\ne 0$ (verdadeira), continua sequencialmente no bloco `THEN`.
3. No final do bloco `THEN`, coloca-se um salto incondicional `JUMP label_fim` para evitar executar o bloco `ELSE`.

```assembly
    // Avaliar condição... (topo fica 1 ou 0)
    jz bloco_else

    // Bloco THEN
    ...
    jump fim_if

bloco_else:
    // Bloco ELSE
    ...

fim_if:
```

---
### 📌 Exemplo Demonstrativo 2: Par ou Ímpar
**Problema:** Ler um inteiro e determinar se é **Par** ou **Ímpar** utilizando o operador `MOD 2`.


In [ ]:
exemplo_2 = """
pushi 0    // gp[0]: numero lido

start
    pushs "Introduza um numero inteiro: "
    writes
    read
    atoi
    storeg 0

    // Testar se numero % 2 == 0
    pushg 0
    pushi 2
    mod
    
    // Se mod == 0, entao e par.
    // jz salta se for zero, logo salta se for par!
    jz e_par

e_impar:
    pushs "O numero e IMPAR."
    writes
    writeln
    jump fim

e_par:
    pushs "O numero e PAR."
    writes
    writeln

fim:
stop
"""

print("Teste 1 (Numero 14):")
ewvm_run(exemplo_2, inputs=["14"])
print("\nTeste 2 (Numero 9):")
ewvm_run(exemplo_2, inputs=["9"])


---
### ✏️ Exercício 2.1: Maior de 2 Números Lidos
**Objetivo:** Ler dois números inteiros $A$ e $B$ do utilizador e imprimir o maior entre eles. Se forem iguais, indicar qualquer um deles.

**Instruções a praticar:** `SUP` ($m > n$), `JZ`, `JUMP`.


In [ ]:
# Espaço para a resolução do Exercício 2.1:
exercicio_2_1 = """
pushi 0    // gp[0]: A
pushi 0    // gp[1]: B

start
    pushs "Introduza o valor A: "
    writes
    read
    atoi
    storeg 0

    pushs "Introduza o valor B: "
    writes
    read
    atoi
    storeg 1

    // Comparar se A > B
    pushg 0
    pushg 1
    sup
    jz b_maior_ou_igual

a_maior:
    pushs "O maior numero e A = "
    writes
    pushg 0
    writei
    writeln
    jump fim_ex2_1

b_maior_ou_igual:
    pushs "O maior numero e B = "
    writes
    pushg 1
    writei
    writeln

fim_ex2_1:
stop
"""

ewvm_run(exercicio_2_1, inputs=["45", "78"])


---
### ✏️ Exercício 2.2: Máximo de 3 Números Lidos ($A, B, C$)
**Objetivo:** Ler três números inteiros $A, B$ e $C$, calcular e imprimir o valor máximo entre eles através de seleções condicionais aninhadas ou de uma variável acumuladora `max`.


In [ ]:
# Espaço para a resolução do Exercício 2.2:
exercicio_2_2 = """
pushi 0    // gp[0]: A
pushi 0    // gp[1]: B
pushi 0    // gp[2]: C
pushi 0    // gp[3]: max

start
    // Leitura de A, B, C
    read
    atoi
    storeg 0

    read
    atoi
    storeg 1

    read
    atoi
    storeg 2

    // Inicializar max = A
    pushg 0
    storeg 3

    // Se B > max entao max = B
    pushg 1
    pushg 3
    sup
    jz check_c
    pushg 1
    storeg 3

check_c:
    // Se C > max entao max = C
    pushg 2
    pushg 3
    sup
    jz print_max
    pushg 2
    storeg 3

print_max:
    pushs "O valor maximo entre os 3 e: "
    writes
    pushg 3
    writei
    writeln
stop
"""

# Teste com A=12, B=89, C=34 (Máximo esperado = 89)
ewvm_run(exercicio_2_2, inputs=["12", "89", "34"])


---
### ✏️ Exercício 2.3: Classificação de Notas Escolares
**Objetivo:** Ler uma nota inteira entre $0$ e $20$ e imprimir a classificação correspondente:
* Nota $\ge 16$: `"Excelente"`
* $10 \le$ Nota $< 16$: `"Aprovado"`
* Nota $< 10$: `"Reprovado"`


In [ ]:
# Espaço para a resolução do Exercício 2.3:
exercicio_2_3 = """
pushi 0    // gp[0]: nota

start
    pushs "Introduza a nota (0-20): "
    writes
    read
    atoi
    storeg 0

    // Testar se nota >= 16
    pushg 0
    pushi 16
    supeq
    jz testar_aprovado

    pushs "Classificacao: Excelente"
    writes
    writeln
    jump fim_nota

testar_aprovado:
    // Testar se nota >= 10
    pushg 0
    pushi 10
    supeq
    jz reprovado

    pushs "Classificacao: Aprovado"
    writes
    writeln
    jump fim_nota

reprovado:
    pushs "Classificacao: Reprovado"
    writes
    writeln

fim_nota:
stop
"""

print("Teste Nota 18:")
ewvm_run(exercicio_2_3, inputs=["18"])
print("Teste Nota 12:")
ewvm_run(exercicio_2_3, inputs=["12"])
print("Teste Nota 7:")
ewvm_run(exercicio_2_3, inputs=["7"])


---
## 🟠 Nível 3: Estruturas de Repetição e Ciclos Iterativos (`WHILE`, `FOR`)

### 🧠 Fundamento Teórico: O Padrão Canónico de um Ciclo
Um ciclo `while (condicao) { corpo; }` é traduzido em Assembly de pilha colocando a etiqueta de início do ciclo antes do teste da condição:
```assembly
ciclo:
    // 1. Avaliar a condição de continuação no ciclo
    ...
    // 2. Se a condição for falsa (0), saltar para fora do ciclo
    jz fim_ciclo

    // 3. Corpo do ciclo (atualização de acumuladores, contadores, etc.)
    ...

    // 4. Saltar de volta ao início do ciclo
    jump ciclo

fim_ciclo:
```

---
### 📌 Exemplo Demonstrativo 3: Somatório de $1$ a $N$
**Problema:** Ler um número inteiro positivo $N$ e calcular a soma:
$$S = \sum_{i=1}^{N} i = 1 + 2 + 3 + \dots + N$$


In [ ]:
exemplo_3 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: i (contador)
pushi 0    // gp[2]: soma (acumulador)

start
    pushs "Introduza N: "
    writes
    read
    atoi
    storeg 0

    // i = 1
    pushi 1
    storeg 1

    // soma = 0
    pushi 0
    storeg 2

ciclo_soma:
    // Condição: i <= N
    pushg 1
    pushg 0
    infeq
    jz fim_soma

    // soma = soma + i
    pushg 2
    pushg 1
    add
    storeg 2

    // i = i + 1
    pushg 1
    pushi 1
    add
    storeg 1

    jump ciclo_soma

fim_soma:
    pushs "Somatorio de 1 a N = "
    writes
    pushg 2
    writei
    writeln
stop
"""

# Somatório de 1 a 10 = 55
ewvm_run(exemplo_3, inputs=["10"])


---
### ✏️ Exercício 3.1: Fatorial de um Número Inteiro ($N!$)
**Objetivo:** Ler um inteiro positivo $N$ e calcular o seu fatorial:
$$N! = N \times (N-1) \times (N-2) \times \dots \times 1$$
*(Convencione $0! = 1$)*.


In [ ]:
# Espaço para a resolução do Exercício 3.1:
exercicio_3_1 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: fat (acumulador de produto)

start
    pushs "Introduza N para calcular N!: "
    writes
    read
    atoi
    storeg 0

    // fat = 1
    pushi 1
    storeg 1

loop_fat:
    // Enquanto N > 1
    pushg 0
    pushi 1
    sup
    jz fim_fat

    // fat = fat * N
    pushg 1
    pushg 0
    mul
    storeg 1

    // N = N - 1
    pushg 0
    pushi 1
    sub
    storeg 0

    jump loop_fat

fim_fat:
    pushs "Fatorial = "
    writes
    pushg 1
    writei
    writeln
stop
"""

# Testar com N = 5 (Esperado = 120)
ewvm_run(exercicio_3_1, inputs=["5"])


---
### ✏️ Exercício 3.2: Potência Inteira ($Base^{Exp}$)
**Objetivo:** Ler dois inteiros positivos $B$ (base) e $E$ (expoente), e calcular $B^E$ através de multiplicações sucessivas sem recorrer a bibliotecas externas.


In [ ]:
# Espaço para a resolução do Exercício 3.2:
exercicio_3_2 = """
pushi 0    // gp[0]: base B
pushi 0    // gp[1]: exp E
pushi 0    // gp[2]: res (acumulador inicializado a 1)

start
    read
    atoi
    storeg 0

    read
    atoi
    storeg 1

    // res = 1
    pushi 1
    storeg 2

loop_pot:
    // Enquanto E > 0
    pushg 1
    pushi 0
    sup
    jz fim_pot

    // res = res * B
    pushg 2
    pushg 0
    mul
    storeg 2

    // E = E - 1
    pushg 1
    pushi 1
    sub
    storeg 1

    jump loop_pot

fim_pot:
    pushs "Resultado da potencia: "
    writes
    pushg 2
    writei
    writeln
stop
"""

# 2^8 = 256
ewvm_run(exercicio_3_2, inputs=["2", "8"])


---
### ✏️ Exercício 3.3: Sequência de Fibonacci (Primeiros $N$ Termos)
**Objetivo:** Ler $N$ e imprimir os primeiros $N$ termos da sequência de Fibonacci:
$$0, 1, 1, 2, 3, 5, 8, 13, 21, \dots$$


In [ ]:
# Espaço para a resolução do Exercício 3.3:
exercicio_3_3 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: a = 0
pushi 0    // gp[2]: b = 1
pushi 0    // gp[3]: temp
pushi 0    // gp[4]: contador i

start
    read
    atoi
    storeg 0

    // Inicializar a=0, b=1, i=0
    pushi 0
    storeg 1
    pushi 1
    storeg 2
    pushi 0
    storeg 4

loop_fib:
    // Enquanto i < N
    pushg 4
    pushg 0
    inf
    jz fim_fib

    // Imprimir a
    pushg 1
    writei
    pushs " "
    writes

    // temp = a + b
    pushg 1
    pushg 2
    add
    storeg 3

    // a = b
    pushg 2
    storeg 1

    // b = temp
    pushg 3
    storeg 2

    // i = i + 1
    pushg 4
    pushi 1
    add
    storeg 4

    jump loop_fib

fim_fib:
    writeln
stop
"""

# Primeiros 8 termos: 0 1 1 2 3 5 8 13
ewvm_run(exercicio_3_3, inputs=["8"])


---
### ✏️ Exercício 3.4: Algoritmo de Euclides para o MDC (Máximo Divisor Comum)
**Objetivo:** Ler dois números inteiros positivos $A$ e $B$ e calcular o seu Máximo Divisor Comum utilizando o algoritmo clássico de Euclides:
$$\text{MDC}(A, B) = \text{MDC}(B, A \pmod B) \quad \text{até } B = 0$$


In [ ]:
# Espaço para a resolução do Exercício 3.4:
exercicio_3_4 = """
pushi 0    // gp[0]: A
pushi 0    // gp[1]: B
pushi 0    // gp[2]: resto

start
    read
    atoi
    storeg 0

    read
    atoi
    storeg 1

loop_mdc:
    // Enquanto B != 0
    pushg 1
    pushi 0
    equal
    not
    jz fim_mdc

    // resto = A % B
    pushg 0
    pushg 1
    mod
    storeg 2

    // A = B
    pushg 1
    storeg 0

    // B = resto
    pushg 2
    storeg 1

    jump loop_mdc

fim_mdc:
    pushs "MDC = "
    writes
    pushg 0
    writei
    writeln
stop
"""

# MDC de 48 e 18 = 6
ewvm_run(exercicio_3_4, inputs=["48", "18"])


---
### ✏️ Exercício 3.5: Contagem e Soma dos Dígitos de um Inteiro
**Objetivo:** Ler um número inteiro positivo (ex: $4528$) e determinar:
1. O número total de dígitos ($4$).
2. A soma de todos os seus dígitos ($4 + 5 + 2 + 8 = 19$).

*Sugestão:* Utilize sucessivas divisões inteiras por $10$ (`DIV 10`) e restos por $10$ (`MOD 10`).


In [ ]:
# Espaço para a resolução do Exercício 3.5:
exercicio_3_5 = """
pushi 0    // gp[0]: num
pushi 0    // gp[1]: conta_digitos
pushi 0    // gp[2]: soma_digitos

start
    read
    atoi
    storeg 0

    pushi 0
    storeg 1
    pushi 0
    storeg 2

loop_dig:
    // Enquanto num > 0
    pushg 0
    pushi 0
    sup
    jz fim_dig

    // soma = soma + (num % 10)
    pushg 2
    pushg 0
    pushi 10
    mod
    add
    storeg 2

    // conta = conta + 1
    pushg 1
    pushi 1
    add
    storeg 1

    // num = num / 10
    pushg 0
    pushi 10
    div
    storeg 0

    jump loop_dig

fim_dig:
    pushs "Contagem de digitos: "
    writes
    pushg 1
    writei
    writeln

    pushs "Soma dos digitos: "
    writes
    pushg 2
    writei
    writeln
stop
"""

# Testar com 4528 (Contagem: 4, Soma: 19)
ewvm_run(exercicio_3_5, inputs=["4528"])


---
## 🔴 Nível 4: Alocação Dinâmica e Estruturas de Dados na Heap (Arrays)

### 🧠 Fundamento Teórico: Manipulação de Arrays na Struct Heap
A EWVM dispõe de uma memória dinâmica estruturada (**Struct Heap**). Para manipular arrays:
1. **Alocação:**
   - `ALLOC n`: Aloca um bloco de $n$ células contíguas e devolve o seu endereço base.
   - `ALLOCN`: Retira o tamanho $n$ da pilha, aloca o bloco e devolve o endereço base.
2. **Escrita (`STOREN` / `STORE`):**
   - Para escrever num índice dinâmico $i$, coloca-se na pilha: `[endereço, índice, valor]` e chama-se `STOREN`.
3. **Leitura (`LOADN` / `LOAD`):**
   - Para ler do índice dinâmico $i$, coloca-se na pilha: `[endereço, índice]` e chama-se `LOADN`.

```
                    Struct Heap (Heap de Estruturas)
            Endereço Base: a
            ┌─────────┬─────────┬─────────┬─────────┐
            │  a[0]   │  a[1]   │  a[2]   │  a[n-1] │
            └─────────┴─────────┴─────────┴─────────┘
              ▲         ▲
   Escrita:   pushg addr; pushg i; pushg valor; storen
   Leitura:   pushg addr; pushg i; loadn
```

---
### 📌 Exemplo Demonstrativo 4: Alocação, Leitura e Soma de um Array
**Problema:** Ler o tamanho $N$ de um array, alocar o vetor na heap, ler os seus $N$ elementos e calcular a soma total.


In [ ]:
exemplo_4 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: Endereco base do array na Heap
pushi 0    // gp[2]: soma acumulada
pushi 0    // gp[3]: contador i

start
    // 1. Ler N
    pushs "Tamanho do array (N): "
    writes
    read
    atoi
    storeg 0

    // 2. Alocar array na Heap de tamanho N
    pushg 0
    allocn
    storeg 1

    // 3. Ciclo de Leitura
    pushi 0
    storeg 3
loop_read:
    pushg 3
    pushg 0
    inf
    jz fim_read

    pushg 1    // Endereço
    pushg 3    // Índice
    read
    atoi       // Valor lido
    storen     // a[i] = valor

    pushg 3
    pushi 1
    add
    storeg 3
    jump loop_read
fim_read:

    // 4. Ciclo de Soma
    pushi 0
    storeg 2    // soma = 0
    pushi 0
    storeg 3    // i = 0
loop_soma:
    pushg 3
    pushg 0
    inf
    jz fim_soma_arr

    // Obter a[i]
    pushg 1    // Endereço
    pushg 3    // Índice
    loadn      // Empilha a[i]

    pushg 2
    add
    storeg 2   // soma = soma + a[i]

    pushg 3
    pushi 1
    add
    storeg 3
    jump loop_soma
fim_soma_arr:

    pushs "Soma dos elementos do array: "
    writes
    pushg 2
    writei
    writeln
stop
"""

# Array com 4 elementos: [10, 25, 5, 100] -> Soma = 140
ewvm_run(exemplo_4, inputs=["4", "10", "25", "5", "100"])


---
### ✏️ Exercício 4.1: Máximo e Mínimo de um Array
**Objetivo:** Alocar um array de $N$ inteiros na Heap, ler os seus elementos e encontrar simultaneamente o **valor máximo** e o **valor mínimo** presentes no array.


In [ ]:
# Espaço para a resolução do Exercício 4.1:
exercicio_4_1 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: arr_addr
pushi 0    // gp[2]: max_val
pushi 0    // gp[3]: min_val
pushi 0    // gp[4]: i

start
    read
    atoi
    storeg 0

    // Alocar N elementos
    pushg 0
    allocn
    storeg 1

    // Ler elementos
    pushi 0
    storeg 4
l_ler:
    pushg 4
    pushg 0
    inf
    jz fim_ler
    pushg 1
    pushg 4
    read
    atoi
    storen
    pushg 4
    pushi 1
    add
    storeg 4
    jump l_ler
fim_ler:

    // Inicializar max_val = arr[0] e min_val = arr[0]
    pushg 1
    pushi 0
    loadn
    storeg 2

    pushg 1
    pushi 0
    loadn
    storeg 3

    // Percorrer de i = 1 ate N-1
    pushi 1
    storeg 4
l_busca:
    pushg 4
    pushg 0
    inf
    jz fim_busca

    // Carregar arr[i]
    pushg 1
    pushg 4
    loadn

    // Testar se arr[i] > max_val
    dup 1
    pushg 2
    sup
    jz test_min
    dup 1
    storeg 2

test_min:
    // Testar se arr[i] < min_val
    dup 1
    pushg 3
    inf
    jz prox_elem
    dup 1
    storeg 3

prox_elem:
    pop 1      // descartar cópia de arr[i]
    pushg 4
    pushi 1
    add
    storeg 4
    jump l_busca

fim_busca:
    pushs "Valor Maximo = "
    writes
    pushg 2
    writei
    writeln

    pushs "Valor Minimo = "
    writes
    pushg 3
    writei
    writeln
stop
"""

# Array de 5 elementos: [30, 7, 89, 12, 45] -> Max=89, Min=7
ewvm_run(exercicio_4_1, inputs=["5", "30", "7", "89", "12", "45"])


---
### ✏️ Exercício 4.2: Inversão *In-Place* de um Array na Heap
**Objetivo:** Ler um array de $N$ elementos e invertê-lo diretamente na memória (*in-place*, sem alocar outro array), trocando a posição $left$ com a posição $right$ enquanto $left < right$. No final, imprimir o array invertido.


In [ ]:
# Espaço para a resolução do Exercício 4.2:
exercicio_4_2 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: addr
pushi 0    // gp[2]: left
pushi 0    // gp[3]: right
pushi 0    // gp[4]: temp_left
pushi 0    // gp[5]: temp_right

start
    read
    atoi
    storeg 0

    pushg 0
    allocn
    storeg 1

    // Ler array
    pushi 0
    storeg 2
l_in_read:
    pushg 2
    pushg 0
    inf
    jz fim_in_read
    pushg 1
    pushg 2
    read
    atoi
    storen
    pushg 2
    pushi 1
    add
    storeg 2
    jump l_in_read
fim_in_read:

    // left = 0, right = N - 1
    pushi 0
    storeg 2
    pushg 0
    pushi 1
    sub
    storeg 3

loop_inv:
    // Enquanto left < right
    pushg 2
    pushg 3
    inf
    jz print_inv

    // temp_left = arr[left]
    pushg 1
    pushg 2
    loadn
    storeg 4

    // temp_right = arr[right]
    pushg 1
    pushg 3
    loadn
    storeg 5

    // arr[left] = temp_right
    pushg 1
    pushg 2
    pushg 5
    storen

    // arr[right] = temp_left
    pushg 1
    pushg 3
    pushg 4
    storen

    // left = left + 1; right = right - 1
    pushg 2
    pushi 1
    add
    storeg 2

    pushg 3
    pushi 1
    sub
    storeg 3

    jump loop_inv

print_inv:
    pushs "Array invertido: "
    writes
    pushi 0
    storeg 2
l_p:
    pushg 2
    pushg 0
    inf
    jz fim_p
    pushg 1
    pushg 2
    loadn
    writei
    pushs " "
    writes
    pushg 2
    pushi 1
    add
    storeg 2
    jump l_p
fim_p:
    writeln
stop
"""

# Array [1, 2, 3, 4, 5] -> invertido: 5 4 3 2 1
ewvm_run(exercicio_4_2, inputs=["5", "1", "2", "3", "4", "5"])


---
### ✏️ Exercício 4.3: Pesquisa Linear com Índice
**Objetivo:** Ler um array de $N$ inteiros e, de seguida, ler um valor chave $K$. Pesquisar $K$ no array e imprimir o índice onde ocorre pela primeira vez. Caso não exista, imprimir $-1$.


In [ ]:
# Espaço para a resolução do Exercício 4.3:
exercicio_4_3 = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: addr
pushi 0    // gp[2]: chave K
pushi 0    // gp[3]: i
pushi 0    // gp[4]: idx_encontrado (-1)

start
    read
    atoi
    storeg 0

    pushg 0
    allocn
    storeg 1

    // Leitura do array
    pushi 0
    storeg 3
l_p_read:
    pushg 3
    pushg 0
    inf
    jz fim_p_read
    pushg 1
    pushg 3
    read
    atoi
    storen
    pushg 3
    pushi 1
    add
    storeg 3
    jump l_p_read
fim_p_read:

    // Leitura da chave K
    read
    atoi
    storeg 2

    // idx_encontrado = -1
    pushi -1
    storeg 4

    // Pesquisa
    pushi 0
    storeg 3
loop_search:
    pushg 3
    pushg 0
    inf
    jz fim_search

    // Comparar arr[i] == K
    pushg 1
    pushg 3
    loadn
    pushg 2
    equal
    jz prox_search

    // Encontrou!
    pushg 3
    storeg 4
    jump fim_search

prox_search:
    pushg 3
    pushi 1
    add
    storeg 3
    jump loop_search

fim_search:
    pushs "Indice da chave no array: "
    writes
    pushg 4
    writei
    writeln
stop
"""

# Array [10, 40, 25, 90, 70], Chave = 25 (Esperado: 2)
ewvm_run(exercicio_4_3, inputs=["5", "10", "40", "25", "90", "70", "25"])


---
## 🟣 Nível 5: Funções, Procedimentos e Recursividade (`CALL`, `RETURN`)

### 🧠 Fundamento Teórico: O Frame de Ativação e a Pilha de Chamadas
O suporte a procedimentos e funções na EWVM baseia-se nas instruções `PUSHA`, `CALL` e `RETURN`:
1. **Passagem de Parâmetros e Chamada:**
   - O chamador empilha os argumentos da função (ex: argumento $x$).
   - O chamador empilha o endereço da função (`pusha nome_funcao`).
   - A instrução `CALL` retira o endereço de destino, guarda na **Call Stack** o endereço de regresso (`PC + 1`) e o `FP` antigo, e atualiza `FP = SP` e `PC = destino`.
2. **Acesso aos Parâmetros dentro da Função:**
   - Como `FP` foi ajustado para o topo da pilha após retirar o endereço do salto, o último argumento empilhado encontra-se exatamente no offset **`-1`** relativo a `FP`!
   - Podemos aceder ao argumento com:
     ```assembly
     pushfp
     load -1       // Carrega o argumento 1 da função
     ```
   - Ou atualizar o argumento (ou valor de retorno) com:
     ```assembly
     storel -1     // Escreve no offset fp[-1]
     ```
3. **Variáveis Locais:**
   - As variáveis locais criadas dentro da função residem a partir de `fp[0]`, `fp[1]`, etc., acessíveis diretamente via `pushl 0` e `storel 0`.
4. **Retorno (`RETURN`):**
   - A instrução `RETURN` restaura `SP = FP` (descartando todas as variáveis locais da função), recupera o antigo `FP` e `PC` da Call Stack e incrementa `PC` para continuar no chamador.

---
### 📌 Exemplo Demonstrativo 5: Função Procedimental `imprime_quadrado(x)`


In [ ]:
exemplo_5 = """
start
    pushs "Programa principal a chamar funcao..."
    writes
    writeln

    // Empilhar argumento = 9
    pushi 9

    // Empilhar endereco e chamar funcao
    pusha quadrado
    call

    pushs "Regresso ao programa principal com sucesso!"
    writes
    writeln
stop

// -------------------------------------------------------------------
// Funcao quadrado(x): imprime x e x^2
// Argumento x esta acessivel em fp[-1]
// -------------------------------------------------------------------
quadrado:
    pushs "Dentro da funcao -> Quadrado de "
    writes
    pushfp
    load -1
    writei
    pushs " e: "
    writes
    pushfp
    load -1
    pushfp
    load -1
    mul
    writei
    writeln
    return
"""

ewvm_run(exemplo_5, trace=True)


---
### ✏️ Exercício 5.1: Função `maximo(a, b)` com Retorno de Valor
**Objetivo:** Implementar uma função que recebe dois números $a$ e $b$ através da pilha, calcula o maior deles e devolve o resultado ao chamador no topo da pilha.


In [ ]:
# Espaço para a resolução do Exercício 5.1:
exercicio_5_1 = """
start
    // Chamar maximo(35, 84)
    pushi 35       // Argumento a: fp[-2]
    pushi 84       // Argumento b: fp[-1]
    pusha func_max
    call

    // O retorno substituiu fp[-2], logo fica no topo da pilha
    pushs "Resultado retornado pela funcao: "
    writes
    writei
    writeln
stop

// -------------------------------------------------------------------
// Funcao func_max(a, b)
// a = fp[-2], b = fp[-1]
// -------------------------------------------------------------------
func_max:
    pushfp
    load -2    // a
    pushfp
    load -1    // b
    sup
    jz b_vence

a_vence:
    // Guarda a no offset -2 (onde ficara o retorno)
    pushfp
    load -2
    storel -2
    jump ret_max

b_vence:
    // Guarda b no offset -2
    pushfp
    load -1
    storel -2

ret_max:
    // Descartar o argumento b para deixar apenas o retorno
    return
"""

ewvm_run(exercicio_5_1,trace=True)


---
### ✏️ Exercício 5.2: Função Recursiva — Fatorial ($N!$)
**Objetivo:** Implementar o cálculo do Fatorial através de uma **função recursiva**:
$$\text{fat}(n) = \begin{cases} 1 & \text{se } n \le 1 \\ n \times \text{fat}(n - 1) & \text{se } n > 1 \end{cases}$$

Cada invocação recursiva gera um novo frame na pilha de chamadas, demonstrando a elegância e poder da arquitetura de pilha da EWVM!


In [ ]:
# Espaço para a resolução do Exercício 5.2:
exercicio_5_2 = """
start
    pushs "Introduza N para fatorial recursivo: "
    writes
    read
    atoi       // Argumento N

    pusha fat_rec
    call

    pushs "Fatorial Recursivo = "
    writes
    writei
    writeln
stop

// -------------------------------------------------------------------
// fat_rec(n): n esta em fp[-1]
// Devolve n! substituindo o valor em fp[-1]
// -------------------------------------------------------------------
fat_rec:
    pushfp
    load -1
    pushi 1
    infeq
    jz passo_recursivo

caso_base:
    pushi 1
    storel -1
    return

passo_recursivo:
    // Chamada recursiva: fat_rec(n - 1)
    pushfp
    load -1
    pushi 1
    sub        // n - 1

    pusha fat_rec
    call       // Retorna (n - 1)! no topo da pilha

    // Multiplicar por n: n * fat_rec(n - 1)
    pushfp
    load -1
    mul

    // Guardar resultado na posicao de retorno fp[-1]
    storel -1
    return
"""

# Fatorial recursivo de 6 = 720
ewvm_run(exercicio_5_2, inputs=["6"])


---
### ✏️ Exercício 5.3: Procedimento Recursivo — Sequência de Fibonacci ($Fib(N)$)
**Objetivo:** Implementar a função recursiva pura de Fibonacci:
$$\text{fib}(n) = \begin{cases} 0 & \text{se } n = 0 \\ 1 & \text{se } n = 1 \\ \text{fib}(n - 1) + \text{fib}(n - 2) & \text{se } n > 1 \end{cases}$$


In [ ]:
# Espaço para a resolução do Exercício 5.3:
exercicio_5_3 = """
start
    pushs "Calcular Fibonacci recursivo de N = "
    writes
    read
    atoi

    pusha fib_rec
    call

    pushs "Resultado: "
    writes
    writei
    writeln
stop

// -------------------------------------------------------------------
// fib_rec(n): n em fp[-1]
// Devolve fib(n) em fp[-1]
// -------------------------------------------------------------------
fib_rec:
    // Caso n == 0
    pushfp
    load -1
    pushi 0
    equal
    jz check_um

    pushi 0
    storel -1
    return

check_um:
    // Caso n == 1
    pushfp
    load -1
    pushi 1
    equal
    jz rec_fib_passo

    pushi 1
    storel -1
    return

rec_fib_passo:
    // 1. fib(n - 1)
    pushfp
    load -1
    pushi 1
    sub
    pusha fib_rec
    call       // Topo tem fib(n - 1)

    // 2. fib(n - 2)
    pushfp
    load -1
    pushi 2
    sub
    pusha fib_rec
    call       // Topo tem fib(n - 2), subtopo tem fib(n - 1)

    // 3. Somar fib(n - 1) + fib(n - 2)
    add

    // Guardar no retorno fp[-1]
    storel -1
    return
"""

# Fibonacci recursivo de 7 = 13
ewvm_run(exercicio_5_3, inputs=["7"])


---
## 🏆 6. Desafio Final Integrador: Ordenação de um Array na Heap (Bubble Sort)

Como consolidação de todos os conceitos aprendidos (aritmética, condicionais, ciclos aninhados e manipulação de memória dinâmica na Heap), apresentamos o algoritmo de **Bubble Sort** para ordenar por ordem crescente uma lista de números fornecida pelo utilizador.


In [ ]:
desafio_bubble_sort = """
pushi 0    // gp[0]: N
pushi 0    // gp[1]: array_addr
pushi 0    // gp[2]: i (ciclo externo)
pushi 0    // gp[3]: j (ciclo interno)
pushi 0    // gp[4]: temp_a
pushi 0    // gp[5]: temp_b

start
    pushs "=== BUBBLE SORT NA EWVM ==="
    writes
    writeln
    pushs "Numero de elementos: "
    writes
    read
    atoi
    storeg 0

    // Alocar array na Heap
    pushg 0
    allocn
    storeg 1

    // Leitura dos N elementos
    pushi 0
    storeg 2
loop_in:
    pushg 2
    pushg 0
    inf
    jz fim_in
    pushg 1
    pushg 2
    read
    atoi
    storen
    pushg 2
    pushi 1
    add
    storeg 2
    jump loop_in
fim_in:

    // Ciclo Externo: i de 0 ate N - 1
    pushi 0
    storeg 2
loop_ext:
    pushg 2
    pushg 0
    pushi 1
    sub
    inf
    jz print_ordenado

    // Ciclo Interno: j de 0 ate N - i - 2
    pushi 0
    storeg 3
loop_int:
    pushg 3
    pushg 0
    pushg 2
    sub
    pushi 1
    sub
    inf
    jz fim_loop_int

    // temp_a = arr[j]
    pushg 1
    pushg 3
    loadn
    storeg 4

    // temp_b = arr[j + 1]
    pushg 1
    pushg 3
    pushi 1
    add
    loadn
    storeg 5

    // Se arr[j] > arr[j + 1] -> Trocar!
    pushg 4
    pushg 5
    sup
    jz sem_troca

    // arr[j] = temp_b
    pushg 1
    pushg 3
    pushg 5
    storen

    // arr[j + 1] = temp_a
    pushg 1
    pushg 3
    pushi 1
    add
    pushg 4
    storen

sem_troca:
    pushg 3
    pushi 1
    add
    storeg 3
    jump loop_int

fim_loop_int:
    pushg 2
    pushi 1
    add
    storeg 2
    jump loop_ext

print_ordenado:
    pushs "Array ordenado: "
    writes
    pushi 0
    storeg 2
loop_out:
    pushg 2
    pushg 0
    inf
    jz fim_out
    pushg 1
    pushg 2
    loadn
    writei
    pushs " "
    writes
    pushg 2
    pushi 1
    add
    storeg 2
    jump loop_out

fim_out:
    writeln
stop
"""

# Testar ordenação de 6 elementos: [64, 34, 25, 12, 22, 11]
# Esperado: 11 12 22 25 34 64
ewvm_run(desafio_bubble_sort, inputs=["6", "64", "34", "25", "12", "22", "11"])


---
## 🎯 Conclusão e Próximos Passos
Parabéns! Concluiu a ficha prática tutorial da **EWVM**.

Com os conhecimentos consolidados nesta ficha:
1. Conhece o funcionamento detalhado de uma **Máquina de Pilha** e o papel crucial de cada registo (`PC`, `SP`, `FP`, `GP`).
2. Sabe gerar instruções de máquina para expressões aritméticas, estruturas de seleção (`IF-THEN-ELSE`), estruturas de repetição (`WHILE`) e arranjos dinâmicos na Heap.
3. Compreende a mecânica da pilha de ativação em chamadas procedimentais e funções recursivas.

Estes padrões constituem a base necessária para a implementação do módulo de **Geração de Código Intermédio / Assembly** do compilador da unidade curricular de **Processamento de Linguagens**!
